# Step 4 — 分類器訓練與評估
載入向量，依時間順序切割訓練/測試集，訓練 NB / KNN / SVM / DT / RF 並輸出混淆矩陣

Big Data & Business Analytics, National Taiwan University

In [28]:
# ── 可調整參數區 ──────────────────────────────────────
TEST_RATIO = 0.2   # 測試資料比例（後 20% 時間段）
# ──────────────────────────────────────────────────────

In [29]:
import csv
import numpy as np
from scipy import sparse
from sklearn.naive_bayes      import MultinomialNB
from sklearn.neighbors        import KNeighborsClassifier
from sklearn.svm              import SVC
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import RandomForestClassifier
from sklearn.metrics          import accuracy_score, confusion_matrix, classification_report

# [METHOD] 目前方法：時序切割 80/20 | 可替換為：移動回測（step5）
# [METHOD] 目前方法：NB/KNN/SVM/DT/RF | 可替換為：投票集成、LLM 零樣本分類

In [30]:
# 載入稀疏矩陣（由 step3 的 limit_model.npz 輸出）
X = sparse.load_npz('limit_model.npz')

# 載入對應標籤（由 step3 的 articles_processed_Ngram.csv 輸出）
y = []
with open('articles_processed_Ngram.csv', newline='', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    for row in reader:
        y.append(int(row['label']))
y = np.array(y)

print(f'共 {X.shape[0]} 篇文章，{X.shape[1]} 維特徵')
print(f'看漲：{sum(y==1)} 篇，看跌：{sum(y==-1)} 篇')

共 2529 篇文章，1564461 維特徵
看漲：1528 篇，看跌：1001 篇


In [31]:
# 依時間順序切割（articles 已按 post_time 排序）
# 不用隨機切割，保持時間先後
split_idx = int(X.shape[0] * (1 - TEST_RATIO))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'訓練集：{X_train.shape[0]} 篇，測試集：{X_test.shape[0]} 篇')

訓練集：2023 篇，測試集：506 篇


In [32]:
# 評估函式（訓練 → 預測 → 印出混淆矩陣、準確率、precision/recall/f1）
def evaluate(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm  = confusion_matrix(y_test, y_pred, labels=[1, -1])

    print(f'\n===== {name} =====')
    print(f'準確率：{acc*100:.1f}%')
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, labels=[1, -1],
                                target_names=['看漲', '看跌']))
    print('Confusion Matrix (列=真實, 欄=預測):')
    print(f'{"":16s} 預測漲  預測跌')
    print(f'{"真實漲":16s}  {cm[0][0]:5d}    {cm[0][1]:5d}')
    print(f'{"真實跌":16s}  {cm[1][0]:5d}    {cm[1][1]:5d}')
    return acc

In [33]:
# Naive Bayes（需要非負輸入，TF 向量天然滿足）
acc_nb = evaluate('Naive Bayes', MultinomialNB(), X_train, y_train, X_test, y_test)


===== Naive Bayes =====
準確率：39.9%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.40      1.00      0.57       202
          看跌       0.00      0.00      0.00       304

    accuracy                           0.40       506
   macro avg       0.20      0.50      0.29       506
weighted avg       0.16      0.40      0.23       506

Confusion Matrix (列=真實, 欄=預測):
                 預測漲  預測跌
真實漲                 202        0
真實跌                 304        0


c:\Users\a3504\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\a3504\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\a3504\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [34]:
# KNN（k=5）
acc_knn = evaluate('KNN (k=5)', KNeighborsClassifier(n_neighbors=5), X_train, y_train, X_test, y_test)


===== KNN (k=5) =====
準確率：40.1%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.37      0.69      0.48       202
          看跌       0.50      0.21      0.29       304

    accuracy                           0.40       506
   macro avg       0.44      0.45      0.39       506
weighted avg       0.45      0.40      0.37       506

Confusion Matrix (列=真實, 欄=預測):
                 預測漲  預測跌
真實漲                 140       62
真實跌                 241       63


In [35]:
# SVM（線性核）
acc_svm = evaluate('SVM', SVC(kernel='linear'), X_train, y_train, X_test, y_test)


===== SVM =====
準確率：38.9%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.39      0.95      0.55       202
          看跌       0.35      0.02      0.04       304

    accuracy                           0.39       506
   macro avg       0.37      0.48      0.30       506
weighted avg       0.37      0.39      0.24       506

Confusion Matrix (列=真實, 欄=預測):
                 預測漲  預測跌
真實漲                 191       11
真實跌                 298        6


In [24]:
# Decision Tree
acc_dt = evaluate('Decision Tree', DecisionTreeClassifier(random_state=42), X_train, y_train, X_test, y_test)


===== Decision Tree =====
準確率：51.4%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.43      0.66      0.52       202
          看跌       0.65      0.41      0.51       304

    accuracy                           0.51       506
   macro avg       0.54      0.54      0.51       506
weighted avg       0.56      0.51      0.51       506

Confusion Matrix (列=真實, 欄=預測):
                 預測漲  預測跌
真實漲                 134       68
真實跌                 178      126


In [25]:
# Random Forest
acc_rf = evaluate('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, X_test, y_test)


===== Random Forest =====
準確率：50.6%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.44      0.82      0.57       202
          看跌       0.71      0.30      0.42       304

    accuracy                           0.51       506
   macro avg       0.57      0.56      0.50       506
weighted avg       0.60      0.51      0.48       506

Confusion Matrix (列=真實, 欄=預測):
                 預測漲  預測跌
真實漲                 165       37
真實跌                 213       91


In [26]:
# 準確率彙整
print('\n===== 準確率彙整 =====')
results = [
    ('Naive Bayes',   acc_nb),
    ('KNN (k=5)',     acc_knn),
    ('SVM',           acc_svm),
    ('Decision Tree', acc_dt),
    ('Random Forest', acc_rf),
]
for name, acc in sorted(results, key=lambda x: x[1], reverse=True):
    print(f'  {name:20s}: {acc*100:.1f}%')


===== 準確率彙整 =====
  Naive Bayes         : 53.2%
  Decision Tree       : 51.4%
  Random Forest       : 50.6%
  SVM                 : 48.6%
  KNN (k=5)           : 45.1%
